# 01 — Exploración de la Spotify API

Objetivo: verificar acceso a la API, explorar los endpoints principales y construir el primer batch del dataset propio.

**Antes de ejecutar**: asegúrate de tener el archivo `.env` con tus credenciales.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.api.spotify_client import get_client
from src.api.extractor import get_audio_features, get_tracks_from_playlist, get_genre_playlists

## 1. Verificar conexión a la API

In [ ]:
sp = get_client()

result = sp.search(q='The Beatles', type='artist', limit=1)
artist = result['artists']['items'][0]
print(f"Conexión OK — Artista: {artist['name']}, Popularidad: {artist['popularity']}")

## 2. Audio features de una canción concreta

In [ ]:
# Bohemian Rhapsody como ejemplo
track_id = '7tFiyTwD0nx5a1eklYtX2J'
features = sp.audio_features([track_id])[0]

audio_cols = ['danceability','energy','key','loudness','speechiness',
              'acousticness','instrumentalness','liveness','valence','tempo']
pd.Series({k: features[k] for k in audio_cols})

## 3. Construir primer dataset desde playlists por género

In [ ]:
GENRES = ['pop', 'rock', 'jazz', 'electronic', 'hip-hop', 'classical', 'latin']

genre_playlists = get_genre_playlists(GENRES, limit_per_genre=3, sp=sp)
print({g: len(v) for g, v in genre_playlists.items()})

In [ ]:
all_tracks = []
for genre, playlist_ids in genre_playlists.items():
    for pid in playlist_ids:
        try:
            df = get_tracks_from_playlist(pid, sp=sp)
            df['genre'] = genre
            all_tracks.append(df)
        except Exception as e:
            print(f'Error en playlist {pid}: {e}')

df_tracks = pd.concat(all_tracks, ignore_index=True).drop_duplicates(subset='id')
print(f'Total canciones únicas: {len(df_tracks)}')
df_tracks.head()

In [ ]:
df_features = get_audio_features(df_tracks['id'].tolist(), sp=sp)
df_dataset = df_tracks.merge(df_features, left_on='id', right_on='id', how='inner')

df_dataset.to_csv('../data/raw/tracks_raw.csv', index=False)
print(f'Dataset guardado: {df_dataset.shape[0]} canciones, {df_dataset.shape[1]} columnas')